<a href="https://colab.research.google.com/github/atessberfin/voltradar/blob/main/notebooks/VoltRadar_Masters_Thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **VoltRadar: Forecasting Import Growth and Monitoring Trade-Policy Interventions in Lithium-Ion Battery Markets**

# **1. Business Problem Understanding**

International market selection is commonly supported by analysing historical trade data to identify countries with growing demand. However, import trends alone may not provide a complete assessment because government interventions, including tariffs, subsidies and import restrictions, can strengthen or weaken a potential market opportunity. In practice, analysts often conduct this process manually by reviewing trade data and subsequently investigating relevant policy developments.

VoltRadar aims to partially automate this process for lithium-ion accumulators (HS 850760). The project will forecast country-level import growth using historical Trade Map data and examine whether the inclusion of structured Global Trade Alert intervention variables improves forecasting performance. The resulting predictions will support an explainable assessment of market growth, policy conditions and potential export opportunities.

# **2. Data Collection and Understanding**

This project uses two secondary datasets. Historical country-level import values for lithium-ion accumulators (HS 850760) were obtained from Trade Map. Trade-policy intervention records affecting the selected product were obtained from Global Trade Alert. The datasets will first be examined separately and will later be cleaned, transformed and integrated at the country-year level for modelling.


## **2.1. Importing Libraries**

In [3]:
!pip install -q xlrd

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## **2.2. Loading the Datasets**

In [7]:
trade_df = pd.read_excel(
    "/content/trade_map_data.xlsx",
    engine="openpyxl"
)

print("Trade Map dataset shape:", trade_df.shape)
display(trade_df.head())

Trade Map dataset shape: (231, 21)


,Importers,Imported value in 2006,Imported value in 2007,Imported value in 2008,Imported value in 2009,Imported value in 2010,Imported value in 2011,Imported value in 2012,Imported value in 2013,Imported value in 2014,Imported value in 2015,Imported value in 2016,Imported value in 2017,Imported value in 2018,Imported value in 2019,Imported value in 2020,Imported value in 2021,Imported value in 2022,Imported value in 2023,Imported value in 2024,Imported value in 2025
0,World,NaN,NaN,NaN,NaN,NaN,NaN,10170016.0,11449019.0,12961890.0,13682466.0,15174344.0,19312728.0,25672564.0,31038055.0,40249915.0,61900015.0,89087370.0,116620002.0,111605857.0,124000680.0
1,Germany,0.0,0.0,0.0,0.0,0.0,0.0,565425.0,676968.0,1031521.0,1313127.0,1614343.0,2247410.0,2857808.0,3707772.0,6406589.0,10358053.0,14411438.0,23313634.0,20051795.0,22974328.0
2,United States of America,0.0,0.0,0.0,0.0,0.0,0.0,1306942.0,1707471.0,1729944.0,1718175.0,1997259.0,2568701.0,3224108.0,3691381.0,4823512.0,8258723.0,13898653.0,18749588.0,23848685.0,20084555.0
3,Viet Nam,0.0,0.0,0.0,0.0,0.0,0.0,198721.0,182078.0,673182.0,668028.0,701292.0,1063209.0,1302762.0,1660844.0,2210261.0,3766166.0,3868655.0,3288450.0,3496012.0,4978530.0
4,"Korea, Republic of",0.0,0.0,0.0,0.0,0.0,0.0,464023.0,511797.0,462385.0,402507.0,394591.0,670951.0,1224303.0,1248932.0,1632726.0,3357021.0,5694928.0,8465963.0,4754948.0,4946958.0


In [8]:
gta_df = pd.read_csv(
    "/content/interventions.csv",
    low_memory=False
)

print("Global Trade Alert dataset shape:", gta_df.shape)
display(gta_df.head())

Global Trade Alert dataset shape: (1605, 19)


,Intervention ID,Intervention URL,State Act ID,State Act URL,State Act Title,GTA Evaluation,Implementing Jurisdictions,Implementation Level,Eligible Firm,Intervention Type,Mast Chapter,Affected Sectors,Affected Products,Affected Jurisdictions,Is Horizontal,Date Announced,Date Implemented,Date Removed,Is In Force
0,11086,https://globaltradealert.org/intervention/11086,3290,https://www.globaltradealert.org/state-act/3290,United States of America: Suspension of Argentina's GSP benefits,Red,United States of America,National,all,Import tariff,Tariff measures,"012, 013, 015, 016, 019, 021, 029, 032, 042, 043, 049, 151, 161, 163, 211, 212, 213, 214, 215, 2...","10631, 10632, 10633, 10639, 20230, 20890, 20910, 20990, 21091, 21092, 21093, 21099, 30245, 30246...","Argentina, Seychelles",False,2012-03-26,2012-05-29,2018-02-28,0
1,11360,https://globaltradealert.org/intervention/11360,4315,https://www.globaltradealert.org/state-act/4315,Lithuania: Liberalizations of the energy sector,Red,Lithuania,National,all,FDI: Entry and ownership rule,FDI measures,"171, 341, 345, 461, 464, 471, 472","271600, 290410, 290420, 290431, 290432, 290433, 290434, 290435, 290436, 290491, 290499, 290511, ...","Croatia, Finland, Hungary, Italy, Latvia, Luxembourg, Russia, Macedonia, United Kingdom",False,2010-10-06,2012-06-26,NaN,1
2,12536,https://globaltradealert.org/intervention/12536,8896,https://www.globaltradealert.org/state-act/8896,Russian Federation: Incentives for Special Investment Contracts in selected industries,Amber,Russia,National,all,Instrument unclear,Instrument unclear,"019, 029, 031, 032, 222, 232, 239, 261, 262, 263, 264, 265, 266, 267, 268, 271, 272, 273, 279, 2...","280110, 280120, 280130, 280200, 280300, 280410, 280421, 280429, 280430, 280440, 280450, 280461, ...","Azerbaijan, Argentina, Australia, Austria, Bahrain, Bangladesh, Armenia, Belgium, Bolivia, Bosni...",False,2015-07-16,NaN,NaN,0
3,13025,https://globaltradealert.org/intervention/13025,10379,https://www.globaltradealert.org/state-act/10379,Malaysia: Introduction of import license requirements for a large number of goods,Red,Malaysia,National,all,Import licensing requirement,"E: Non-automatic licensing, quotas etc.","019, 231, 232, 235, 250, 282, 341, 342, 352, 369, 391, 392, 393, 411, 412, 429, 435, 441, 445, 4...","110100, 170112, 170113, 170114, 170191, 170199, 170230, 170240, 170260, 170290, 240110, 240120, ...","Argentina, Australia, Austria, Bangladesh, Belgium, Brazil, Bulgaria, Myanmar, Cambodia, Canada,...",False,2013-03-01,2013-03-01,NaN,1
4,13055,https://globaltradealert.org/intervention/13055,10410,https://www.globaltradealert.org/state-act/10410,"EC: Fiji, Iraq, Marshall Islands, Samoa & Tonga scrapped from GSP",Red,"Austria, Belgium, Bulgaria, Croatia, Cyprus, Czechia, Denmark, Estonia, Finland, France, Germany...",Supranational,all,Import tariff,Tariff measures,"012, 015, 018, 019, 023, 029, 213, 214, 222, 223, 231, 233, 234, 236, 239, 241, 242, 243, 244, 2...","40510, 40520, 40590, 40711, 40719, 40721, 40729, 40790, 40900, 60210, 60220, 60230, 60240, 60290...","Fiji, Iraq, Marshall Islands, Tonga, Samoa",False,2015-11-05,2017-01-01,NaN,1


## **2.3. Dataset Overview**

In [9]:
dataset_overview = pd.DataFrame({
    "Dataset": [
        "Trade Map",
        "Global Trade Alert"
    ],
    "Rows": [
        trade_df.shape[0],
        gta_df.shape[0]
    ],
    "Columns": [
        trade_df.shape[1],
        gta_df.shape[1]
    ]
})

display(dataset_overview)

print("\nTrade Map columns:")
print(trade_df.columns.tolist())

print("\nGlobal Trade Alert columns:")
print(gta_df.columns.tolist())

,Dataset,Rows,Columns
0,Trade Map,231,21
1,Global Trade Alert,1605,19



Trade Map columns:
['Importers', 'Imported value in 2006', 'Imported value in 2007', 'Imported value in 2008', 'Imported value in 2009', 'Imported value in 2010', 'Imported value in 2011', 'Imported value in 2012', 'Imported value in 2013', 'Imported value in 2014', 'Imported value in 2015', 'Imported value in 2016', 'Imported value in 2017', 'Imported value in 2018', 'Imported value in 2019', 'Imported value in 2020', 'Imported value in 2021', 'Imported value in 2022', 'Imported value in 2023', 'Imported value in 2024', 'Imported value in 2025']

Global Trade Alert columns:
['Intervention ID', 'Intervention URL', 'State Act ID', 'State Act URL', 'State Act Title', 'GTA Evaluation', 'Implementing Jurisdictions', 'Implementation Level', 'Eligible Firm', 'Intervention Type', 'Mast Chapter', 'Affected Sectors', 'Affected Products', 'Affected Jurisdictions', 'Is Horizontal', 'Date Announced', 'Date Implemented', 'Date Removed', 'Is In Force']


In [10]:
def create_column_overview(dataframe):
    return pd.DataFrame({
        "Data Type": dataframe.dtypes.astype(str),
        "Non-Null Values": dataframe.notna().sum(),
        "Missing Values": dataframe.isna().sum(),
        "Unique Values": dataframe.nunique(dropna=True)
    })


print("Trade Map column overview:")
display(create_column_overview(trade_df))

print("\nGlobal Trade Alert column overview:")
display(create_column_overview(gta_df))

Trade Map column overview:


,Data Type,Non-Null Values,Missing Values,Unique Values
Importers,object,231,0,231
Imported value in 2006,float64,162,69,1
Imported value in 2007,float64,169,62,1
Imported value in 2008,float64,169,62,1
Imported value in 2009,float64,169,62,1
Imported value in 2010,float64,170,61,1
Imported value in 2011,float64,170,61,1
Imported value in 2012,float64,209,22,116
Imported value in 2013,float64,211,20,142
Imported value in 2014,float64,210,21,161



Global Trade Alert column overview:


,Data Type,Non-Null Values,Missing Values,Unique Values
Intervention ID,int64,1605,0,1605
Intervention URL,object,1605,0,1605
State Act ID,int64,1605,0,1272
State Act URL,object,1605,0,1272
State Act Title,object,1605,0,1267
GTA Evaluation,object,1605,0,3
Implementing Jurisdictions,object,1605,0,107
Implementation Level,object,1605,0,5
Eligible Firm,object,1605,0,7
Intervention Type,object,1605,0,56


In [11]:
trade_years = [
    int(column.split()[-1])
    for column in trade_df.columns
    if column.startswith("Imported value in")
]

implemented_dates = pd.to_datetime(
    gta_df["Date Implemented"],
    errors="coerce"
)

print("Trade Map year range:", min(trade_years), "-", max(trade_years))
print("Number of importers:", trade_df["Importers"].nunique())

print("\nGTA implementation date range:")
print(implemented_dates.min(), "-", implemented_dates.max())

print("\nGTA Evaluation distribution:")
print(gta_df["GTA Evaluation"].value_counts(dropna=False))

print("\nDuplicate Intervention IDs:")
print(gta_df["Intervention ID"].duplicated().sum())

Trade Map year range: 2006 - 2025
Number of importers: 231

GTA implementation date range:
2012-01-01 00:00:00 - 2026-11-10 00:00:00

GTA Evaluation distribution:
GTA Evaluation
Red      1159
Green     323
Amber     123
Name: count, dtype: int64

Duplicate Intervention IDs:
0


# **3. Exploratory Data Analysis (EDA)**

## **3.1. Trade Map Data Analysis**

## **3.2. Global Trade Alert Data Analysis**

## **3.3. Initial Findings**

# **4. Data Cleaning and Preprocessing**

## **4.1. Trade Map Data Cleaning**

## **4.2. GTA Data Cleaning**

## **4.3. Final Data Quality Check**

# **5. Feature Engineering and Data Integration**

## **5.1. Trade Features**

## **5.2. GTA Features**

## **5.3. Dataset Integration**

## **5.4. Target Variable**

## **5.5. Final Modelling Dataset**

# **6. Model Training**

## **6.1. Time-Based Data Split**

## **6.2. Baseline Model**

## **6.3. Model 1: Trade-Only Forecast**

## **6.4. Model 2: Policy-Enriched Forecast**

## **6.5. Hyperparameter Tuning**

# **7. Model Evaluation and Comparison**

## **7.1. Evaluation Metrics**

## **7.2. Model 1 Results**

## **7.3. Model 2 Results**

## **7.4. Model Comparison**

# **8. Interpretation and Visualisation**

## **8.1. Feature Importance**

## **8.2. SHAP Analysis**

## **8.3. Actual vs Predicted Values**

## **8.4. Country-Level Predictions**

## **8.5. Market Opportunity Output**

# **9. Model Export and Prototype Preparation**

## **9.1. Final Model Selection**

## **9.2. Model Export**

## **9.3. Data Export**

## **9.4. Prediction Function**

# **10. Conclusion**